# ORPO: Odds Ratio Preference Optimization

**ORPO** (Odds Ratio Preference Optimization) is a preference alignment technique that combines **supervised fine-tuning (SFT)** and **preference alignment into a single training step**. Unlike methods such as RLHF or DPO, which require a separate SFT stage followed by preference optimization, ORPO does both simultaneously by modifying the standard language modeling loss.

### How ORPO Works

The key insight behind ORPO is simple: the standard cross-entropy loss used in SFT **doesn't distinguish** between preferred and dispreferred responses — it just maximizes the likelihood of whatever text it sees. ORPO adds an **odds ratio–based penalty** to the loss function that:

1. **Increases** the likelihood of generating **chosen** (preferred) responses
2. **Decreases** the likelihood of generating **rejected** (dispreferred) responses

The ORPO loss is:

$$\mathcal{L}_{ORPO} = \mathcal{L}_{SFT} + \lambda \cdot \mathcal{L}_{OR}$$

Where $\mathcal{L}_{SFT}$ is the standard cross-entropy loss on the **chosen** response, and $\mathcal{L}_{OR}$ is the odds ratio loss that contrasts chosen vs. rejected responses. The hyperparameter $\lambda$ (called `beta` in the TRL implementation) controls how strongly the model is pushed to prefer chosen over rejected completions.

### Why ORPO?

| Aspect | RLHF | DPO | ORPO |
|--------|------|-----|------|
| Reference model needed? | Yes | Yes | **No** |
| Separate SFT stage? | Yes | Yes | **No** |
| Reward model needed? | Yes | No | **No** |
| Training stages | 3 (SFT → RM → PPO) | 2 (SFT → DPO) | **1** |

ORPO is the most streamlined of the three: one model, one stage, one loss function.

### What You Need: A Preference Dataset

Unlike SFT (which only needs prompt + completion pairs), ORPO requires a **preference dataset** with three columns:

- `prompt`: the user query / instruction
- `chosen`: the preferred (high-quality) response
- `rejected`: the dispreferred (lower-quality) response

The model learns to assign higher probability to `chosen` responses and lower probability to `rejected` ones — all in a single pass.

### Setup

For better reproducibility during training, use the pinned versions below, the same versions used in the course:

In [1]:
!pip install transformers==4.56.1 peft==0.17.0 accelerate==1.10.0 trl==0.23.1 bitsandbytes==0.47.0 datasets==4.0.0 huggingface-hub==0.34.4 safetensors==0.6.2 pandas==2.2.2 matplotlib==3.10.0 numpy==2.0.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.9/503.9 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.7/374.7 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.6/564.6 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.8/485.8 kB 17.7 MB/s eta 0:00:00
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.7.0
    Uninstalling safetensors-0.7.0:
      Successfully uninstalled safetensors-0.7.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.8.0
    Uninstalling huggingface_hub-1.8.0:
      Successfully uninstalled huggingface_hub-1.8

### Imports

In [2]:
import os
import torch
from contextlib import nullcontext
from datasets import load_dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import ORPOConfig, ORPOTrainer

Notice the key difference from QLoRA/SFT: we import `ORPOConfig` and `ORPOTrainer` instead of `SFTConfig` and `SFTTrainer`. Also, we **don't** need `get_peft_model` — the `ORPOTrainer` handles LoRA/PEFT setup internally when you pass a `peft_config`.

## Loading a Quantized Base Model

Just like in the QLoRA notebook, we start by loading a 4-bit quantized model using `BitsAndBytesConfig`. This keeps the base model small in GPU memory while we attach trainable LoRA adapters on top.

We'll use **Phi-3 Mini 4K Instruct** — the same model from the QLoRA notebook — so you can directly compare how the same base model behaves under SFT vs. ORPO.

In [9]:
bnb_config = BitsAndBytesConfig(
   load_in_4bit=True,                    # load weights as 4-bit
   bnb_4bit_quant_type="nf4",            # use NormalFloat4 quantization
   bnb_4bit_use_double_quant=True,        # quantize the quantization constants too
   bnb_4bit_compute_dtype=torch.float32   # compute in FP32 for stability
)

repo_id = 'microsoft/Phi-3-mini-4k-instruct'

model = AutoModelForCausalLM.from_pretrained(
    repo_id,
    device_map="cuda:0",
    quantization_config=bnb_config
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [10]:
print(f"Model memory footprint: {model.get_memory_footprint()/1e6:.2f} MB")

Model memory footprint: 2206.34 MB


Even in 4-bit, the model takes about 2+ GB of VRAM. The quantized `Linear4bit` layers can't be trained directly — that's where LoRA adapters come in.

In the QLoRA notebook, we manually called `prepare_model_for_kbit_training()` and then `get_peft_model()`. With `ORPOTrainer`, we can skip that — it handles PEFT setup internally when we pass a `peft_config` argument. We only need to define the LoRA configuration.

## Setting Up the LoRA Configuration

We define the LoRA configuration the same way as in QLoRA. The `ORPOTrainer` will use this config to:
1. Call `prepare_model_for_kbit_training()` on the quantized model
2. Attach LoRA adapters to the specified target modules

So we **don't** need to manually call `get_peft_model()` — the trainer does it for us.

In [11]:
peft_config = LoraConfig(
    r=8,                   # rank of the adapter — lower = fewer trainable parameters
    lora_alpha=16,         # scaling factor, usually 2*r
    bias="none",           # don't train biases — avoids modifying base model behavior
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    # Phi-3 requires manually specifying target modules
    target_modules=['o_proj', 'qkv_proj', 'gate_up_proj', 'down_proj'],
)

The LoRA config is identical to what we used in the QLoRA notebook. The difference is **when and how** it gets applied:

| | QLoRA + SFT | QLoRA + ORPO |
|---|---|---|
| LoRA setup | Manual: `prepare_model_for_kbit_training()` → `get_peft_model(model, config)` | Automatic: pass `peft_config` to `ORPOTrainer` |
| Model passed to trainer | Already wrapped PeftModel | Raw quantized model |
| Trainer class | `SFTTrainer` | `ORPOTrainer` |

## Loading the Preference Dataset

This is where ORPO fundamentally differs from SFT. Instead of prompt/completion pairs, we need a **preference dataset** with `prompt`, `chosen`, and `rejected` columns.

We'll use the [`trl-lib/ultrafeedback_binarized`](https://huggingface.co/datasets/trl-lib/ultrafeedback_binarized) dataset — a well-known preference dataset where:
- **`prompt`**: a user instruction or question
- **`chosen`**: the response rated as higher quality
- **`rejected`**: the response rated as lower quality

Both `chosen` and `rejected` are in **conversational format** (list of `{role, content}` dicts), which is exactly what `ORPOTrainer` expects.

In [12]:
dataset = load_dataset("trl-lib/ultrafeedback_binarized", split="train")
dataset

Dataset({
    features: ['chosen', 'rejected', 'score_chosen', 'score_rejected'],
    num_rows: 62135
})

Let's inspect a single example to understand the structure:

In [14]:
example = dataset[0]
print("=== CHOSEN (first 300 chars) ===")
print(str(example['chosen'])[:300])
print("\n=== REJECTED (first 300 chars) ===")
print(str(example['rejected'])[:300])

=== CHOSEN (first 300 chars) ===
[{'content': 'Use the pygame library to write a version of the classic game Snake, with a unique twist', 'role': 'user'}, {'content': "Sure, I'd be happy to help you write a version of the classic game Snake using the pygame library! Here's a basic outline of how we can approach this:\n\n1. First, w

=== REJECTED (first 300 chars) ===
[{'content': 'Use the pygame library to write a version of the classic game Snake, with a unique twist', 'role': 'user'}, {'content': 'Sure, here\'s an example of how to write a version of Snake game with a unique twist using the Pygame library:\n```python\nimport pygame\n\nclass SnakeGame:\n    def


Each example has the same prompt but two different responses — one preferred (`chosen`) and one not (`rejected`). The ORPO loss uses both during training to simultaneously:
1. Teach the model *what* to say (SFT on `chosen`)
2. Teach the model *what not* to say (odds ratio penalty using `chosen` vs `rejected`)

Since UltraFeedback is a large dataset (~61k examples) and we're running on a consumer GPU, let's take a subset for training:

In [15]:
# Take a manageable subset for consumer GPU training
# Feel free to increase this if you have more VRAM / time
dataset = dataset.shuffle(seed=42).select(range(1000))
print(f"Training on {len(dataset)} examples")

Training on 1000 examples


### Tokenizer

We load the tokenizer corresponding to our model, just like in the QLoRA notebook. The tokenizer's chat template determines how the conversational messages get formatted into the final token sequence.

In [16]:
tokenizer = AutoTokenizer.from_pretrained(repo_id)
tokenizer.pad_token = tokenizer.unk_token
tokenizer.pad_token_id = tokenizer.unk_token_id

print("Pad token:", tokenizer.pad_token)
print("Chat template available:", tokenizer.chat_template is not None)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

Pad token: <unk>
Chat template available: True


Let's see how the tokenizer formats a conversation using its chat template:

In [17]:
sample_messages = [
    {"role": "user", "content": "What is ORPO?"},
    {"role": "assistant", "content": "ORPO stands for Odds Ratio Preference Optimization."}
]
print(tokenizer.apply_chat_template(sample_messages, tokenize=False))

<|user|>
What is ORPO?<|end|>
<|assistant|>
ORPO stands for Odds Ratio Preference Optimization.<|end|>
<|endoftext|>


The `ORPOTrainer` will automatically apply this chat template to the `chosen` and `rejected` conversations during preprocessing.

## Configuring ORPO Training

The `ORPOConfig` extends `TrainingArguments` with ORPO-specific parameters. The most important ORPO-specific parameter is **`beta`** — this controls the strength of the odds ratio penalty.

- **Higher `beta`** → stronger preference signal → model more aggressively favors `chosen` over `rejected`
- **Lower `beta`** → weaker preference signal → training behaves more like standard SFT

A typical starting value is `beta=0.1`.

Other parameters like `max_length`, `max_prompt_length`, and `max_completion_length` control how the prompt and completions are truncated/padded during data collation.

In [18]:
orpo_config = ORPOConfig(
    ## ORPO-specific
    beta=0.1,                  # strength of the odds ratio penalty
    max_length=512,            # max total length (prompt + completion)
    max_prompt_length=256,     # max length for the prompt portion

    ## Memory usage
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    gradient_accumulation_steps=2,
    per_device_train_batch_size=2,
    auto_find_batch_size=True,

    ## Training hyperparameters
    num_train_epochs=1,
    learning_rate=5e-5,
    optim='paged_adamw_8bit',
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,

    ## Logging
    logging_steps=10,
    logging_dir='./logs',
    output_dir='./phi3-mini-orpo-adapter',
    report_to='none',

    # Use bf16 if available, otherwise fp16
    bf16=torch.cuda.is_bf16_supported(including_emulation=False),

    # Required by ORPOTrainer
    remove_unused_columns=False,
)

**Key differences from `SFTConfig`:**

| Parameter | SFTConfig | ORPOConfig |
|-----------|-----------|------------|
| `beta` | N/A | Controls odds ratio penalty strength |
| `max_length` | Max sequence length for packing | Max total length (prompt + completion) |
| `max_prompt_length` | N/A | Max prompt portion length |
| `packing` | Supported | Not applicable (each example = one prompt with chosen + rejected) |
| `remove_unused_columns` | Default True | Must be set to **False** |

The `remove_unused_columns=False` is crucial — `ORPOTrainer` needs all columns (`prompt`, `chosen`, `rejected`) during training, and setting this to True would drop them.

We also use a lower learning rate (`5e-5`) compared to the QLoRA SFT notebook (`3e-4`). Preference optimization methods generally benefit from more conservative learning rates since they're optimizing a more complex loss landscape.

## Training with ORPOTrainer

Now we bring everything together. Notice that we pass `peft_config` directly to the trainer — it will handle all the PEFT/LoRA setup internally.

In [19]:
trainer = ORPOTrainer(
    model=model,                    # raw quantized model (NOT a PeftModel)
    args=orpo_config,
    processing_class=tokenizer,
    train_dataset=dataset,
    peft_config=peft_config,        # LoRA config — trainer applies it automatically
)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Let's check the trainable parameters — should be a tiny fraction of the total, just like in QLoRA:

In [20]:
trainable_params = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in trainer.model.parameters())
print(f'Trainable parameters:  {trainable_params/1e6:.2f}M')
print(f'Total parameters:      {total_params/1e6:.2f}M')
print(f'Trainable %:           {100 * trainable_params / total_params:.4f}%')

Trainable parameters:  12.58M
Total parameters:      2021.72M
Trainable %:           0.6224%


Now let's train! During training, you'll see ORPO-specific metrics in the logs:

- **`rewards/chosen`**: mean log-probability of chosen responses (scaled by beta) — should increase
- **`rewards/rejected`**: mean log-probability of rejected responses (scaled by beta) — should decrease
- **`rewards/margins`**: difference between chosen and rejected rewards — should increase
- **`rewards/accuracies`**: how often chosen reward > rejected reward — should approach 1.0
- **`log_odds_ratio`**: the core ORPO signal — should increase

In [21]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.


Step,Training Loss
10,1.530100
20,1.556600
30,1.480500
40,1.384600
50,1.530400
60,1.497500
70,1.384200
80,1.384000
90,1.334100
100,1.346100


TrainOutput(global_step=250, training_loss=1.392352550506592, metrics={'train_runtime': 6430.4784, 'train_samples_per_second': 0.156, 'train_steps_per_second': 0.039, 'total_flos': 0.0, 'train_loss': 1.392352550506592, 'epoch': 1.0})

## Querying the Model

After training, let's test the model. We'll use the same helper functions from the QLoRA notebook to generate prompts and completions.

In [22]:
def gen_prompt(tokenizer, message):
    """Format a user message using the model's chat template."""
    converted_sample = [
        {"role": "user", "content": message},
    ]
    prompt = tokenizer.apply_chat_template(
        converted_sample,
        tokenize=False,
        add_generation_prompt=True
    )
    return prompt


def generate(model, tokenizer, prompt, max_new_tokens=256, skip_special_tokens=True):
    """Generate a completion given a formatted prompt."""
    tokenized_input = tokenizer(
        prompt, add_special_tokens=False, return_tensors="pt"
    ).to(model.device)

    model.eval()
    ctx = (
        torch.autocast(device_type=model.device.type, dtype=model.dtype)
        if model.dtype in [torch.float16, torch.bfloat16]
        else nullcontext()
    )
    with ctx:
        generation_output = model.generate(
            **tokenized_input,
            eos_token_id=tokenizer.eos_token_id,
            max_new_tokens=max_new_tokens
        )

    output = tokenizer.batch_decode(
        generation_output, skip_special_tokens=skip_special_tokens
    )
    return output[0]

In [23]:
# Test with a question from a domain the preference data covers
prompt = gen_prompt(tokenizer, "Explain the difference between a list and a tuple in Python.")
print(generate(trainer.model, tokenizer, prompt))

Explain the difference between a list and a tuple in Python. In Python, both lists and tuples are used to store collections of items. However, there are some key differences between the two:

1. Mutability: Lists are mutable, which means that their contents can be changed after they are created. Tuples, on the other hand, are immutable, which means that their contents cannot be changed once they are created.

2. Syntax: Lists are created using square brackets [], while tuples are created using parentheses ().

3. Performance: Tuples are generally faster than lists because they are immutable. This means that the Python interpreter can optimize the memory usage and access time for tuples.

4. Use cases: Lists are typically used when you need to store a collection of items that may change over time, such as a list of numbers or a list of strings. Tuples are typically used when you need to store a collection of items that will not change, such as a list of coordinates or a list of dates.



In [ ]:
# Try another one
prompt = gen_prompt(tokenizer, "What are the benefits of exercise?")
print(generate(trainer.model, tokenizer, prompt))

The ORPO-trained model should produce more helpful, well-structured responses compared to the base model — not just because it's been fine-tuned on good examples (SFT effect), but because it's been explicitly trained to **prefer** higher-quality responses over lower-quality ones (preference alignment effect).

## Saving the Adapter

Just like with QLoRA, the trained adapter is small and portable. Save it to disk:

In [ ]:
trainer.save_model('local-phi3-mini-orpo-adapter')

In [ ]:
os.listdir('local-phi3-mini-orpo-adapter')

The saved files include the adapter configuration and weights (`adapter_config.json` + `adapter_model.safetensors`), the tokenizer files, and training arguments. The adapter is typically around 50 MB — tiny compared to the full 3.8B parameter model.

### Pushing to the Hub (Optional)

To share your adapter on the Hugging Face Hub:

In [ ]:
from huggingface_hub import login
login()

In [ ]:
trainer.push_to_hub()

## Recap: QLoRA + SFT vs. QLoRA + ORPO

Here's a side-by-side comparison of what changed and what stayed the same:

| Aspect | QLoRA + SFT (Basic_QLoRA notebook) | QLoRA + ORPO (this notebook) |
|--------|-------------------------------------|------------------------------|
| **Base model** | Phi-3 Mini 4K Instruct (4-bit) | Phi-3 Mini 4K Instruct (4-bit) |
| **Quantization** | BitsAndBytesConfig (NF4) | BitsAndBytesConfig (NF4) |
| **LoRA config** | Same r=8, alpha=16 | Same r=8, alpha=16 |
| **PEFT setup** | Manual (`prepare_model` → `get_peft_model`) | Automatic (via `peft_config` arg) |
| **Dataset format** | Conversational: `messages` | Preference: `prompt` + `chosen` + `rejected` |
| **Trainer** | `SFTTrainer` + `SFTConfig` | `ORPOTrainer` + `ORPOConfig` |
| **Loss function** | Cross-entropy only | Cross-entropy + Odds Ratio penalty |
| **Reference model** | N/A | Not needed (unlike DPO) |
| **Key hyperparameter** | — | `beta` (odds ratio strength) |
| **What model learns** | "Generate text like this" | "Generate text like *this*, not like *that*" |

The core quantization + LoRA infrastructure is identical — the difference is entirely in the **training objective** and the **dataset format**. ORPO adds preference awareness to the training loop, teaching the model not just what good outputs look like, but explicitly contrasting them against bad outputs.

That's a wrap! You've fine-tuned a model using ORPO — combining SFT and preference alignment in a single, efficient training step.